In [0]:
spark.sql(
    "DROP TABLE IF EXISTS workspace.gold.dim_network"
)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_network (
    network_key BIGINT,
    network_code STRING
)
""")

In [0]:
network = (
    spark.table(
        "workspace.silver.usgs_earthquakes"
    )
    .select("network")
    .fillna({"network": "UNKNOWN"})
    .dropDuplicates()
    .withColumn(
        "network_key",
        F.xxhash64("network")
    )
    .select(
        "network_key",
        F.col("network").alias("network_code")
    )
)

In [0]:
network.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.gold.dim_network")